In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
from data_pipeline.pipeline import DataPipeline

# .env 파일 로드
env_path = Path("./configs/.env")
load_dotenv(dotenv_path=env_path)

True

In [2]:
# 환경변수 확인
pipeline = DataPipeline.from_env()
print(pipeline.config.roboflow.api_key)
print(pipeline.config.ftp.password)
print(pipeline.config.ssh.password)

Hci8cXDA4hpgvUQwnPFz
waff!23
waff!23


In [3]:
# collection & filter
source_dir = r'C:\Users\User\Desktop\Project\VISION_AI\hdi'
for dir in os.listdir(source_dir):
    pipeline.config.collection.source_subdir = os.path.join(source_dir, dir)
    collection = pipeline.run_collection()
    filtered = pipeline.run_filter(collection)

[11:50:19] pipeline             INFO    ============================================================
[11:50:19] pipeline             INFO    ① 이미지 수집
[11:50:19] collector            INFO    스캔 시작: C:\Users\User\Desktop\Project\VISION_AI\hdi\20260401
[11:50:19] collector            INFO    발견된 파일: 150개
[11:50:19] pipeline             INFO    ============================================================
[11:50:19] pipeline             INFO    ② 필터링 + ③ Dataset 복사
[11:50:20] filter_copy          INFO    복사 28개 / 스킵 122개 / 실패 0개  → C:\Users\User\Desktop\Project\VISION_AI\hdi\dataset
[11:50:20] pipeline             INFO    ============================================================
[11:50:20] pipeline             INFO    ① 이미지 수집
[11:50:20] collector            INFO    스캔 시작: C:\Users\User\Desktop\Project\VISION_AI\hdi\20260402
[11:50:20] collector            INFO    발견된 파일: 194개
[11:50:20] pipeline             INFO    ============================================================
[11:50:20] 

In [ ]:
#upload
dest_path = Path(r'C:\Users\User\Desktop\Project\VISION_AI\hdi\dataset')
pipeline.uploader.config.project_name = 'quarry-x-vision-ai'
pipeline.uploader.config.api_key = 'Hci8cXDA4hpgvUQwnPFz'
pipeline.uploader.simple_upload(dest_path)

[16:02:22] roboflow_upload      INFO    Roboflow 일괄 업로드 시작: C:\Users\User\Desktop\Project\VISION_AI\hdi\dataset


ModuleNotFoundError: No module named 'roboflow'

In [3]:
#download
pipeline.downloader.config.download_version = 2
pipeline.downloader.config.download_location = r'C:\Users\User\Desktop\Project\VISION_AI\dataset'
print('version:', pipeline.downloader.config.download_version)
print('format:', pipeline.downloader.config.download_format)
print('location:', pipeline.downloader.config.download_location)
print('project_name:', pipeline.downloader.config.project_name)
download = pipeline.run_download()

version: 2
format: yolo26
location: C:\Users\User\Desktop\Project\VISION_AI\dataset
project_name: quarry-x-vision-ai
[14:23:46] pipeline             INFO    ============================================================
[14:23:46] pipeline             INFO    ⑥ YOLO 데이터셋 다운로드
[14:23:46] roboflow_download    INFO    다운로드 시작: quarry-x-vision-ai v2 (yolo26)
loading Roboflow workspace...
loading Roboflow project...


Extracting Dataset Version Zip to C:\Users\User\Desktop\Project\VISION_AI\dataset in yolo26:: 100%|██████████| 1498/1498 [00:03<00:00, 405.18it/s]

[14:24:02] roboflow_download    INFO    다운로드 완료: C:\Users\User\Desktop\Project\VISION_AI\dataset


In [4]:
download.location

WindowsPath('C:/Users/User/Desktop/Project/VISION_AI/dataset')

In [12]:
#replace
dest_path = Path(r'C:\Users\User\Desktop\Project\VISION_AI\hdi\dataset')
pipeline.run_replace(download, dest_path)

[12:00:33] pipeline             INFO    ============================================================
[12:00:33] pipeline             INFO    ⑦ 원본 이미지 교체 + 라벨 정리
[12:00:33] image_replacer       INFO    원본 소스: C:\Users\User\Desktop\Project\VISION_AI\hdi\dataset
[12:00:33] image_replacer       INFO    데이터셋:   C:\Users\User\Desktop\Project\VISION_AI\dataset
[12:00:33] image_replacer       INFO    [TRAIN] 원본 교체 + 라벨 정리 시작
[12:00:38] image_replacer       INFO      ✓ train: 교체 595개, 라벨 변경 595개, 원본 누락 0개
[12:00:38] image_replacer       INFO    [VALID] 원본 교체 + 라벨 정리 시작
[12:00:39] image_replacer       INFO      ✓ valid: 교체 68개, 라벨 변경 68개, 원본 누락 0개
[12:00:39] image_replacer       DEBUG   subset 없음: C:\Users\User\Desktop\Project\VISION_AI\dataset\test
[12:00:39] image_replacer       INFO    전체 완료 — 교체 663개, 원본 누락 0개


[ReplaceResult(subset='train', replaced=595, label_renamed=595, missing_source=[]),
 ReplaceResult(subset='valid', replaced=68, label_renamed=68, missing_source=[]),
 ReplaceResult(subset='test', replaced=0, label_renamed=0, missing_source=[])]

In [5]:
#sync
#pipeline.run_sync(download)
pipeline.ftp_sync.upload_directory(local_root=download.location)

[14:24:09] ftp_transfer         INFO    FTP 업로드 시작: 1497개 파일 → /ai_vision/yolo


Upload:   0%|          | 0/1497 [00:00<?, ?file/s]

[14:24:10] ftp_transfer         DEBUG   FTP 연결: administrator@192.168.1.179:21


Upload: 100%|██████████| 1497/1497 [05:42<00:00,  4.38file/s]

[14:29:52] ftp_transfer         INFO    FTP 완료 — 성공 1497, 실패 0 (17.8 MB 전송)


FTPTransferResult(host='192.168.1.179', remote_base='/ai_vision/yolo', transferred=['/ai_vision/yolo/data.yaml', '/ai_vision/yolo/README.dataset.txt', '/ai_vision/yolo/README.roboflow.txt', '/ai_vision/yolo/train/images/1774997331475da0ebf534b9c4532b9b09d0b2f22c8dc_BMP.rf.e0afb4293738da84a0d33ca1b0c89df6.jpg', '/ai_vision/yolo/train/images/1774997434586890edc3a23ab4004b5e67e003bce5c60_BMP.rf.7c72e004cb335c4e401ea34eaea01386.jpg', '/ai_vision/yolo/train/images/17750003906876007a68a245b467d9c080d41ba14f685_BMP.rf.d0566d42dabc857ef5ed7d9a5ad7a986.jpg', '/ai_vision/yolo/train/images/1775000442016670d684bf34b4510bcca757f75eea218_BMP.rf.9ca36de435b0b56af2a02792673b13c5.jpg', '/ai_vision/yolo/train/images/17750034610700c4219d817c64ecbbbe60317a52dbdeb_BMP.rf.e0c50db58ffccccb40f45f4b0cbab2f8.jpg', '/ai_vision/yolo/train/images/177500511813877276541f1904e82be221506ae7fb243_BMP.rf.b062df11fe1d42304ef2b0b6403dd482.jpg', '/ai_vision/yolo/train/images/1775005487966a719bb99b89c4af38008760801176341_BM

In [6]:
pipeline.run_training()

[15:48:21] pipeline             INFO    ============================================================
[15:48:21] pipeline             INFO    ⑧ SSH 원격 학습 트리거
[15:48:22] ssh_training         INFO    SSH 접속: administrator@192.168.1.179
[15:48:22] ssh_training         DEBUG   실행 명령: powershell.exe -Command "Set-Location 'D:\ai_vision\yolo'; & 'D:\ai_vision\yolo\venv\Scripts\python.exe' 'train.py'"
[15:48:28] ssh_training         INFO      | New https://pypi.org/project/ultralytics/8.4.47 available  Update with 'pip install -U ultralytics'
[15:48:29] ssh_training         INFO      | Ultralytics 8.4.40  Python-3.11.9 torch-2.13.0.dev20260420+cu126 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
[15:48:29] ssh_training         INFO      | engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip

TrainingResult(host='192.168.1.179', command='powershell.exe -Command "Set-Location \'D:\\ai_vision\\yolo\'; & \'D:\\ai_vision\\yolo\\venv\\Scripts\\python.exe\' \'train.py\'"', exit_status=0, stdout="New https://pypi.org/project/ultralytics/8.4.47 available  Update with 'pip install -U ultralytics'\r\nUltralytics 8.4.40  Python-3.11.9 torch-2.13.0.dev20260420+cu126 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)\r\n\x1b\x1bengine\\trainer: \x1bagnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, 